In [1]:
from pathlib import Path

dir_path = Path("/scratch/yag1/ego4d_data/v2/clips")
file_names = [p.name for p in dir_path.iterdir() if p.is_file() and p.suffix == ".mp4"] # List of all .mp4 file names in the directory
file_names = [name[:-4] for name in file_names]  # Remove the .mp4 extension

In [2]:
len(file_names)

42

In [3]:
file_names = set(file_names)
len(file_names)

42

In [4]:
import json

In [5]:
with open('/scratch/yag1/ego4d_data/v2/annotations/nlq_val.json', 'r') as f:
    data = json.load(f)


In [6]:
import re

In [7]:
query_df_data = []

In [8]:
for video in data.get("videos", []):
    for clip in video.get("clips", []):
        clip_id = clip.get("clip_uid")
        if clip_id not in file_names:
            continue

        for annotation in clip.get("annotations", []):
            for query_data in annotation.get("language_queries", []):
                query = query_data.get("query", "").lower()

                # Check if the word appears in the query
                if re.search(r"\btalk\b", query, re.IGNORECASE):
                    query_start_sec = query_data.get("clip_start_sec", 0)
                    query_end_sec = query_data.get("clip_end_sec", 0)
                    query_duration = query_end_sec - query_start_sec
                    query_df_data.append({
                        "clip_id": clip_id,
                        "query": query,
                        "query_start_sec": query_start_sec,
                        "query_end_sec": query_end_sec,
                        "query_duration": query_duration
                    })

In [9]:
num_unique_clips = len(set(item["clip_id"] for item in query_df_data))
print(f"Number of unique clips with 'talk' in queries: {num_unique_clips}")

Number of unique clips with 'talk' in queries: 42


In [10]:
print(f"Total queries containing 'talk': {len(query_df_data)}")

Total queries containing 'talk': 55


In [11]:
import pandas as pd

In [12]:
query_df = pd.DataFrame(query_df_data)

In [13]:
query_df.head()

,clip_id,query,query_start_sec,query_end_sec,query_duration
0,a325ce85-cae5-4faa-99bb-7272918fcf19,when did i talk to the cashier at the store,403.38680,408.57537,5.18857
1,a325ce85-cae5-4faa-99bb-7272918fcf19,who did i talk to in the supermarket,404.36400,405.51063,1.14663
2,b69e6150-0309-4202-bf4a-9a342f80d6d7,who did i talk to in the kitchen?,84.48200,88.18600,3.70400
3,35cd9ace-642f-4550-8e63-a5c2caae89ed,who did i talk to in the workshop?,176.98861,180.98800,3.99939
4,35cd9ace-642f-4550-8e63-a5c2caae89ed,who did i talk to in the workshop?,301.55537,323.55500,21.99963


In [14]:
query_df.to_csv("/scratch/yag1/ego4d_data/v2/queries_with_talk.csv", index=False)